In [11]:
# Import necessary libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from sklearn.model_selection import train_test_split
import os

In [27]:
# Define the base directory for the dataset
base_dir = '/kaggle/input/nthu-dataset-ddd-multi-class/Multi class/train'

# Parameters
BATCH_SIZE = 32
IMG_HEIGHT, IMG_WIDTH = 197, 197
num_classes = 4
epochs_initial = 10
epochs_finetune = 30

In [13]:
# List to store data
data = []

# Define categories for drowsiness and not drowsy images
categories = {
    "sleepyCombination": os.path.join(base_dir, "drowsy", "sleepyCombination"),
    "slowBlinkWithNodding": os.path.join(base_dir, "drowsy", "slowBlinkWithNodding"),
    "yawning": os.path.join(base_dir, "drowsy", "yawning"),
    "notdrowsy": os.path.join(base_dir, "notdrowsy")
}


In [14]:

# # Traverse each category and add image paths and labels to the list
# for condition, path in categories.items():
#     if os.path.exists(path):  # Check if the folder exists
#         for image_name in os.listdir(path):
#             image_path = os.path.join(path, image_name)
#             if os.path.isfile(image_path):  # Ensure it is a file
#                 data.append([image_path, condition])

# # Create DataFrame from the collected data
# df = pd.DataFrame(data, columns=["pathofImage", "condition"])

# # Save the DataFrame to a CSV file
# output_csv_path = "dataset.csv"
# df.to_csv(output_csv_path, index=False)

In [15]:
# read from csv
df= pd.read_csv("/kaggle/input/driver-drowsiness/dataset.csv")

In [16]:
df.head()

,pathofImage,condition
0,/kaggle/input/nthu-dataset-ddd-multi-class/Mul...,sleepyCombination
1,/kaggle/input/nthu-dataset-ddd-multi-class/Mul...,sleepyCombination
2,/kaggle/input/nthu-dataset-ddd-multi-class/Mul...,sleepyCombination
3,/kaggle/input/nthu-dataset-ddd-multi-class/Mul...,sleepyCombination
4,/kaggle/input/nthu-dataset-ddd-multi-class/Mul...,sleepyCombination


In [17]:
# Count unique values and their occurrences
unique_counts = df['condition'].value_counts()

# Display the unique values and their counts
print("Unique values and their counts in 'condition' column:")
print(unique_counts)

# Get the number of unique values
num_unique = df['condition'].nunique()
print("\nNumber of unique values in 'condition' column:", num_unique)


Unique values and their counts in 'condition' column:
condition
notdrowsy               30491
sleepyCombination       17756
slowBlinkWithNodding     9412
yawning                  8862
Name: count, dtype: int64

Number of unique values in 'condition' column: 4


In [18]:
# Splitting the data
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['condition'], random_state=42)
test_df, val_df = train_test_split(temp_df, test_size=0.35, stratify=temp_df['condition'], random_state=42)

# Number of rows
print("Number of rows in training set:", len(train_df))
print("Number of rows in validation set:", len(val_df))
print("Number of rows in test set:", len(test_df))

Number of rows in training set: 46564
Number of rows in validation set: 6985
Number of rows in test set: 12972


In [19]:
# Data Generators
train_datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, rotation_range=30, zoom_range=0.3)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    train_df, x_col='pathofImage', y_col='condition', target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE, class_mode='categorical')

val_generator = val_test_datagen.flow_from_dataframe(
    val_df, x_col='pathofImage', y_col='condition', target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE, class_mode='categorical')

test_generator = val_test_datagen.flow_from_dataframe(
    test_df, x_col='pathofImage', y_col='condition', target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)


Found 46564 validated image filenames belonging to 4 classes.
Found 6985 validated image filenames belonging to 4 classes.
Found 12972 validated image filenames belonging to 4 classes.


In [20]:
# Load the Xception model with pre-trained ImageNet weights
# Set `include_top=False` to remove the classification layers
base_model = Xception(weights='imagenet', include_top=False, input_shape=(IMG_WIDTH, IMG_HEIGHT, 3))

# Freeze the base model layers to train only the top layers first
base_model.trainable = False


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [21]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

# Compile the model with frozen base model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Set up TensorBoard for monitoring
tensorboard_callback = TensorBoard(log_dir="logs/xception_initial")


In [22]:
# Initial training with the base model frozen
history_initial = model.fit(
    train_generator,
    epochs=epochs_initial,
    validation_data=val_generator,
    callbacks=[tensorboard_callback]
)

Epoch 1/10


/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1731246291.719447     104 service.cc:145] XLA service 0x7933040039f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1731246291.719504     104 service.cc:153]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1731246300.320206     104 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1456/1456 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step - accuracy: 0.5127 - loss: 1.1434

/opt/conda/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1456/1456 ━━━━━━━━━━━━━━━━━━━━ 703s 471ms/step - accuracy: 0.5127 - loss: 1.1433 - val_accuracy: 0.6457 - val_loss: 0.8239
Epoch 2/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 529s 362ms/step - accuracy: 0.6180 - loss: 0.8837 - val_accuracy: 0.6364 - val_loss: 0.7777
Epoch 3/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 502s 343ms/step - accuracy: 0.6510 - loss: 0.8164 - val_accuracy: 0.7147 - val_loss: 0.6754
Epoch 4/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 504s 345ms/step - accuracy: 0.6801 - loss: 0.7475 - val_accuracy: 0.7009 - val_loss: 0.6881
Epoch 5/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 506s 346ms/step - accuracy: 0.6986 - loss: 0.7123 - val_accuracy: 0.7432 - val_loss: 0.6188
Epoch 6/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 505s 345ms/step - accuracy: 0.7057 - loss: 0.6939 - val_accuracy: 0.7508 - val_loss: 0.5754
Epoch 7/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 511s 349ms/step - accuracy: 0.7152 - loss: 0.6686 - val_accuracy: 0.7679 - val_loss: 0.5691
Epoch 8/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 544s 372ms/step - accuracy: 0.7

In [23]:
# Save the fine-tuned model
model.save('model_xception.h5')

In [24]:
# Fine-tuning: Unfreeze some of the deeper layers in the base model
# Start from block 13 and 14 to allow the model to learn more complex patterns
for layer in base_model.layers:
    if layer.name.startswith('block13') or layer.name.startswith('block14'):
        layer.trainable = True

# Compile with a lower learning rate and momentum for fine-tuning
model.compile(optimizer=optimizers.SGD(learning_rate=1e-4, momentum=0.9),
              loss='categorical_crossentropy', metrics=['accuracy'])

# Set up early stopping to avoid overfitting during fine-tuning
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)


In [28]:
# Fine-tune the model with unfrozen layers
history_finetune = model.fit(
    train_generator,
    epochs=epochs_finetune,
    validation_data=val_generator,
    callbacks=[tensorboard_callback, early_stopping]
)

# Save the fine-tuned model
model.save('fine_tuned_xception_2.h5')


Epoch 1/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 556s 380ms/step - accuracy: 0.8394 - loss: 0.3871 - val_accuracy: 0.8939 - val_loss: 0.2763
Epoch 2/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 496s 339ms/step - accuracy: 0.8391 - loss: 0.3836 - val_accuracy: 0.8966 - val_loss: 0.2743
Epoch 3/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 486s 333ms/step - accuracy: 0.8467 - loss: 0.3696 - val_accuracy: 0.8982 - val_loss: 0.2682
Epoch 4/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 509s 348ms/step - accuracy: 0.8444 - loss: 0.3758 - val_accuracy: 0.9038 - val_loss: 0.2585
Epoch 5/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 510s 349ms/step - accuracy: 0.8490 - loss: 0.3606 - val_accuracy: 0.9018 - val_loss: 0.2548
Epoch 6/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 502s 343ms/step - accuracy: 0.8531 - loss: 0.3532 - val_accuracy: 0.9018 - val_loss: 0.2556
Epoch 7/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 502s 343ms/step - accuracy: 0.8529 - loss: 0.3524 - val_accuracy: 0.9038 - val_loss: 0.2504
Epoch 8/10
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 502s 343ms/step - ac

In [26]:
# Evaluate the model on the validation set
val_loss, val_acc = model.evaluate(val_generator)
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4%}")

219/219 ━━━━━━━━━━━━━━━━━━━━ 21s 96ms/step - accuracy: 0.8977 - loss: 0.2744
Validation Loss: 0.2809, Validation Accuracy: 89.6922%
